# Start

## Import libraries

In [41]:
import numpy as np
import pandas as pd
import sys
import os
import kaggle
import matplotlib.pyplot as plt
from pathlib import Path
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display

## Read CSV and Describe

In [42]:
df = pd.read_csv(r"dataset\202401-202603.csv")

df.describe().apply(lambda s: s.apply('{0:.6f}'.format))

C:\Users\kiwib\AppData\Local\Temp\ipykernel_11616\3790501135.py:1: DtypeWarning: Columns (0: extraction_date_time) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(r"dataset\202401-202603.csv")


,main.aqi,components.co,components.no,components.no2,components.o3,components.so2,components.pm2_5,components.pm10,components.nh3,coord.lon,coord.lat
count,1770901.000000,1770901.000000,1770901.000000,1770901.000000,1770901.000000,1770901.000000,1770901.000000,1770901.000000,1770901.000000,408960.000000,408960.000000
mean,1.482084,259.148498,0.342146,3.177365,45.588174,1.677827,-2337559114.438890,-10129422853.243526,2.124569,122.524569,12.550874
std,0.702584,232.293136,2.834321,88.540088,27.377057,88.410385,152865305014.429108,318090468169.700195,4.433358,1.730648,3.182242
min,1.000000,67.680000,0.000000,-9999.000000,0.000000,-9999.000000,-9998999486464.000000,-9998999486464.000000,0.000000,118.733300,6.087200
25%,1.000000,142.000000,0.000000,0.780000,28.270000,0.350000,2.340000,3.340000,0.210000,120.966700,10.106100
50%,1.000000,202.690000,0.020000,1.840000,40.500000,0.950000,4.630000,6.500000,0.710000,122.550000,13.358600
75%,2.000000,293.730000,0.100000,4.540000,59.370000,2.610000,8.990000,12.300000,2.090000,123.890700,14.767800
max,5.000000,6942.750000,202.060000,235.800000,640.870000,160.220000,373.110000,424.180000,151.550000,126.198600,18.198900


# Cleaning

## Find num of negative vals

 Common Causes of Negative Values
- **Sensor errors or calibration issues**
Some monitoring devices output negative values when they malfunction or drift out of calibration.
- **Missing data encoded as sentinel values**
Many datasets use extreme negative numbers (like -9999) to represent missing or unavailable measurements. For example, your components.no2 and components.so2 have -9999.000000, which is a classic placeholder for "no data."
- **Data preprocessing artifacts**
If raw sensor signals are adjusted (e.g., baseline subtraction, noise filtering), the resulting values can sometimes dip below zero, especially when pollutant levels are near detection limits.
- **Coordinate anomalies**
The huge negative values in components.pm2_5 and components.pm10 (-9998999486464.000000) are clearly not real concentrations. They’re almost certainly corrupted entries or placeholders for missing data.


In [43]:
pollutant_cols = [
    "components.co", "components.no", "components.no2", "components.o3",
    "components.so2", "components.pm2_5", "components.pm10", "components.nh3"
]

# Step 1: Create a mask for rows with any negative values
negative_mask = (df[pollutant_cols] < 0).any(axis=1)

# Step 2: Count total rows with negatives
total_negative_rows = negative_mask.sum()
print("Total rows with negative values:", total_negative_rows)

# Step 3: Count negatives per column
negative_counts = (df[pollutant_cols] < 0).sum()
print("Negative counts per column:\n", negative_counts)

Total rows with negative values: 1932
Negative counts per column:
 components.co          0
components.no          0
components.no2       138
components.o3          0
components.so2       138
components.pm2_5     414
components.pm10     1794
components.nh3         0
dtype: int64


In [44]:
pollutant_cols = [
    "components.co", "components.no", "components.no2", "components.o3",
    "components.so2", "components.pm2_5", "components.pm10", "components.nh3"
]

df[pollutant_cols] = df[pollutant_cols].clip(lower=0)

In [45]:
df.describe().apply(lambda s: s.apply('{0:.6f}'.format))

,main.aqi,components.co,components.no,components.no2,components.o3,components.so2,components.pm2_5,components.pm10,components.nh3,coord.lon,coord.lat
count,1770901.000000,1770901.000000,1770901.000000,1770901.000000,1770901.000000,1770901.000000,1770901.000000,1770901.000000,1770901.000000,408960.000000,408960.000000
mean,1.482084,259.148498,0.342146,3.956551,45.588174,2.457013,7.949147,10.437967,2.124569,122.524569,12.550874
std,0.702584,232.293136,2.834321,6.534461,27.377057,4.699474,11.808724,14.287396,4.433358,1.730648,3.182242
min,1.000000,67.680000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,118.733300,6.087200
25%,1.000000,142.000000,0.000000,0.780000,28.270000,0.350000,2.340000,3.340000,0.210000,120.966700,10.106100
50%,1.000000,202.690000,0.020000,1.840000,40.500000,0.950000,4.630000,6.500000,0.710000,122.550000,13.358600
75%,2.000000,293.730000,0.100000,4.540000,59.370000,2.610000,8.990000,12.300000,2.090000,123.890700,14.767800
max,5.000000,6942.750000,202.060000,235.800000,640.870000,160.220000,373.110000,424.180000,151.550000,126.198600,18.198900


# New DF

In [46]:
aqi_df = pd.DataFrame()
aqi_df['date'] = pd.to_datetime(df['datetime']).dt.date
aqi_df['city'] = df['city_name']
aqi_df['og_aqi'] = df['main.aqi']

# Agg: Recompute AQI

In [47]:
# Breakpoints dictionary for each pollutant

breakpoints = {
    "so2": [
        (0, 20, 0, 50),
        (20, 80, 50, 100),
        (80, 250, 100, 200),
        (250, 350, 200, 300),
        (350, float("inf"), 300, 500),
    ],
    "no2": [
        (0, 40, 0, 50),
        (40, 70, 50, 100),
        (70, 150, 100, 200),
        (150, 200, 200, 300),
        (200, float("inf"), 300, 500),
    ],
    "pm10": [
        (0, 20, 0, 50),
        (20, 50, 50, 100),
        (50, 100, 100, 200),
        (100, 200, 200, 300),
        (200, float("inf"), 300, 500),
    ],
    "pm25": [
        (0, 10, 0, 50),
        (10, 25, 50, 100),
        (25, 50, 100, 200),
        (50, 75, 200, 300),
        (75, float("inf"), 300, 500),
    ],
    "o3": [
        (0, 60, 0, 50),
        (60, 100, 50, 100),
        (100, 140, 100, 200),
        (140, 180, 200, 300),
        (180, float("inf"), 300, 500),
    ],
    "co": [
        (0, 4400, 0, 50),
        (4400, 9400, 50, 100),
        (9400, 12400, 100, 200),
        (12400, 15400, 200, 300),
        (15400, float("inf"), 300, 500),
    ]
}

In [48]:
def compute_category(value, pollutant):
    """Compute AQI sub-index for a given pollutant concentration."""
    if value is None:
        return None
    
    for c_low, c_high, i_low, i_high in breakpoints[pollutant]:
        if c_low <= value <= c_high:
            # Linear interpolation
            aqi = ((i_high - i_low) / (c_high - c_low)) * (value - c_low) + i_low
            return round(aqi)
    return None

In [49]:
def compute_aqi(row):
    # Compute sub-indices for each pollutant
    indices = [
        compute_category(row["components.so2"], "so2"),
        compute_category(row["components.no2"], "no2"),
        compute_category(row["components.pm10"], "pm10"),
        compute_category(row["components.pm2_5"], "pm25"),
        compute_category(row["components.o3"], "o3"),
        compute_category(row["components.co"], "co"),
    ]

    # Filter out None values
    indices = [i for i in indices if i is not None]

    # AQI = maximum index (worst pollutant)
    return max(indices) if indices else None

# Apply to DataFrame
aqi_df["AQI"] = df.apply(compute_aqi, axis=1)

In [50]:
aqi_df.describe().apply(lambda s: s.apply('{0:.6f}'.format))

,og_aqi,AQI
count,1770901.000000,1770901.000000
mean,1.482084,53.299915
std,0.702584,39.048942
min,1.000000,2.000000
25%,1.000000,30.000000
50%,1.000000,41.000000
75%,2.000000,64.000000
max,5.000000,300.000000


## Check accuracy of recomputation

## Separate the og and new aqis in a new df

In [51]:
aqi_only_df = pd.DataFrame()
aqi_only_df['old'] = aqi_df['og_aqi']
aqi_only_df['new'] = aqi_df['AQI']

In [52]:
# Step 1: Define category ranges (example: scale 1–5 mapped to intervals)
# Adjust ranges to your actual categorization rules
category_ranges = {
    1.0: (0, 50),
    2.0: (50, 100),
    3.0: (100, 200),
    4.0: (200, 300),
    5.0: (300, 500)
}

# Step 2: Function to check if computed value falls in the correct range
def check_match(row):
    # Round 'old' to nearest category if it's not exactly 1–5
    cat = round(row["old"])
    if cat not in category_ranges:
        return 0  # treat as mismatch if outside defined categories
    low, high = category_ranges[float(cat)]
    return int(low <= row["new"] <= high)

# Step 3: Apply function
aqi_only_df["match"] = aqi_only_df.apply(check_match, axis=1)

# Step 4: Count matches
total_matches = aqi_only_df["match"].sum()
print(aqi_only_df)
print("Total matches:", total_matches)

KeyboardInterrupt: 

In [ ]:
def check_match(row):
    # Round 'old' to nearest category if it's not exactly 1–5
    cat = round(row["old"])
    if cat not in category_ranges:
        return "invalid"  # outside defined categories
    
    low, high = category_ranges[float(cat)]
    
    if low <= row["new"] <= high:
        return "match"
    elif row["new"] < low:
        return "too low"
    else:  # row["new"] > high
        return "too high"
    
aqi_only_df["match_status"] = aqi_only_df.apply(check_match, axis=1)
mismatches = aqi_only_df[aqi_only_df["match_status"] != "match"]

print(mismatches[["old", "new", "match_status"]])

          old  new match_status
43818   1.000   73     too high
43821   1.000   60     too high
43823   1.000   65     too high
43825   1.000   73     too high
43829   1.000   66     too high
...       ...  ...          ...
735321  3.000   39      too low
735327  2.000    7      too low
735333  1.778   14      too low
735335  2.000   27      too low
735338  4.000   64      too low

[98 rows x 3 columns]


# Pollution Hotspot Index

## Prep DF with needed data

In [60]:
phi_df = pd.DataFrame()
phi_df['date'] = pd.to_datetime(df['datetime']).dt.date
phi_df['city'] = df['city_name']
phi_df['aqi'] = aqi_df['AQI']

phi_df.head()

,date,city,aqi
0,2024-01-01,Alaminos,74
1,2024-01-01,Angeles City,61
2,2024-01-01,Antipolo,23
3,2024-01-01,Bacolod,37
4,2024-01-01,Bacoor,155


## Code to Compute

In [61]:
def compute_phi_per_city(df, threshold):
    """
    Compute Pollution Hotspot Index (PHI_c) for each city.
    
    Parameters:
        df (pd.DataFrame): Must contain 'city' and 'AQI' columns
        threshold (int or float): AQI threshold T
    
    Returns:
        pd.DataFrame: City-level PHI values
    """
    results = (
        df.groupby("city")
          .apply(lambda g: (g["AQI"] > threshold).sum() / len(g))
          .reset_index(name=f"PHI_T{threshold}")
    )
    return results

In [62]:
city_phi = compute_phi_per_city(aqi_df, threshold=150)
print(city_phi)

               city  PHI_T150
0          Alaminos  0.030393
1      Angeles City  0.052636
2          Antipolo  0.072730
3           Bacolod  0.006938
4            Bacoor  0.115770
..              ...       ...
133        Valencia  0.001090
134      Valenzuela  0.096387
135       Victorias  0.006616
136           Vigan  0.014638
137  Zamboanga City  0.000856

[138 rows x 2 columns]


In [63]:
# Sort cities by PHI descending (highest hotspots first)
city_phi_sorted = city_phi.sort_values(by=f"PHI_T150", ascending=False).reset_index(drop=True)

print(city_phi_sorted.head(10))  # top 10 cities

               city  PHI_T150
0             Pasig  0.118488
1       Makati City  0.118225
2    Paranaque City  0.118110
3            Taguig  0.118050
4  Mandaluyong City  0.118005
5            Bacoor  0.115770
6         Las Piñas  0.115466
7       Cavite City  0.115462
8              Imus  0.115453
9      Palayan City  0.108664


In [64]:
# Sort cities by PHI ascending (lowest hotspots first)
city_phi_sorted = city_phi.sort_values(by=f"PHI_T150", ascending=True).reset_index(drop=True)

print(city_phi_sorted.head(10))  # bottom 10 cities

             city  PHI_T150
0        Cotabato  0.000000
1  General Santos  0.000000
2           Digos  0.000000
3          Tandag  0.000000
4         Lamitan  0.000156
5    Surigao City  0.000311
6            Mati  0.000311
7         Dipolog  0.000389
8     Ozamiz City  0.000545
9        Pagadian  0.000545


# Standard Deviation

In [65]:
def compute_std_per_city(df):
    """
    Compute standard deviation of AQI per city.
    
    Parameters:
        df (pd.DataFrame): Must contain 'city' and 'AQI' columns
    
    Returns:
        pd.DataFrame: City-level standard deviation values
    """
    results = (
        df.groupby("city")["aqi"]
          .std(ddof=0)  # population standard deviation (divide by n)
          .reset_index(name="AQI_std")
    )
    return results

In [66]:
city_std = compute_std_per_city(phi_df)
print(city_std.sort_values(by="AQI_std", ascending=False).head(10))

                 city    AQI_std
98              Pasig  69.207038
77        Makati City  69.166649
122            Taguig  69.115734
81   Mandaluyong City  69.114442
97     Paranaque City  69.113511
4              Bacoor  66.505882
35        Cavite City  66.085555
60               Imus  66.083645
70          Las Piñas  66.073123
95       Palayan City  61.430699
